# Tarefa Final - Controlador MIMO

> Felipe Andrade Garcia Tommaselli (11800910)

# Definição do Problema

Neste projeto, consideramos um robô móvel de pequeno porte equipado com motores independentes. O objetivo é projetar um controlador MIMO que permita ao robô seguir uma trajetóriia reta com precisão, apesar de perturbações externas e variações nas características do terreno. Essa abordagem é inspirada em aplicações de robótica off-road, onde a interação do robô com o ambiente é complexa e de difícil modelagem. Em especial, optei por apresentar o problema da **Incerteza Paramétrica**, onde os parâmetros do modelo do robô podem variar dentro de certos limites devido a mudanças no terreno ou desgaste dos componentes.

| Parametro | Representação | Significado | Interpretação Prática |
|-----------|--------|------------------|-------------------|
| Derrapagem/skid angular | ν             | Razão entre a velocidade angular real e do comando   | Derrapamento de roda, lama, solo solto... |
| Derrapagem/skid linear  | μ             | Razão entre a velocidade linear real e do comando    | Deslizamento do contato, atrito do terreno... |

Para esse projeto, adotamos a simulação e os módulos descrito em: https://link.springer.com/article/10.1007/s10846-025-02219-2

A simulação será realizado pelo Gazebo, onde os coeficientes do solo foram alterados para representar essa derrapagem de forma simplificada. 
O resultado final obtido foi um controlador MIMO robusto, capaz de manter o desempenho desejado mesmo diante das incertezas paramétricas introduzidas no sistema: 

![Resultado](demo.gif)

obs: os marcadores verdes foram utilizados como referência de trajetória (linha reta)

# Projeto do Controlador

## 1. Estrutura NΔ ((aula 14b))
Inspirado na aula 14b do curso:

```text
(w, u) ─────▶ (z, y)
        │
        ▼
        Δ
```
Onde:

*   **N:** Sistema em malha fechada nominal (planta + controlador)
*   **Δ:** Bloco de incerteza estruturada (representando as variações em μ, ν)
*   **w:** Entradas exógenas (distúrbios, referência)
*   **z:** Saídas reguladas (erro de rastreamento, esforço de controle)

### Incerteza
Modelamos os parâmetros de derrapagem (*skid*) como:

$$G_p(s) = G_{nom}(s) \cdot (I + W_I(s) \cdot \Delta_I)$$

Onde:

*   $G_{nom}(s)$: Função de transferência nominal da planta
*   $W_I(s)$: Filtro de ponderação da incerteza
*   $\Delta_I$: Perturbação normalizada com $|\Delta_I|_\infty \leq 1$

Peso de Incerteza:

$$W_I(s) = \frac{s/\omega_b + r_0/r_\infty}{s/\omega_b + 1} \cdot r_\infty$$

Com os parâmetros:

*   $r_0 = 0.1$ 
*   $r_\infty = 0.5$ 
*   $\omega_b = 10$ rad/s 

### Robustez

**Estabilidade Robusta (RS)**
O sistema permanece estável para todo $|\Delta|_\infty \leq 1$:

$$\text{RS} \iff \mu_\Delta(N_{11}(j\omega)) < 1, \quad \forall \omega$$

**Desempenho Robusto (RP)**
As especificações de desempenho são mantidas apesar da incerteza:

$$\text{RP} \iff \mu_{\tilde{\Delta}}(N(j\omega)) < 1, \quad \forall \omega$$

Onde $\tilde{\Delta} = \text{diag}\{\Delta, \Delta_P\}$ inclui um bloco de desempenho fictício.

## 2. Modelagem da Planta

Para o controlador implementado, linearizamos as equações do modelo do uniciclo adaptado em torno do equilíbrio:

*   $\theta_{eq} = 0$ (alinhado com a linha)
*   $v_{eq} = v_{ref}$ (velocidade de avanço constante)

Dinâmica Linearizada:

$$\dot{y} = v_{ref} \cdot \theta$$
$$\dot{\theta} = \nu \cdot \omega$$

Isso resulta na representação em espaço de estados:

$$A = \begin{bmatrix} 0 & v_{ref} \\ 0 & 0 \end{bmatrix}, \quad B = \begin{bmatrix} 0 \\ \nu \end{bmatrix}$$

$$C = I_2, \quad D = 0$$


#### Função de Transferência MIMO

De $\omega$ para $[y, \theta]^T$:

$$G(s) = \begin{bmatrix} \frac{v_{ref} \cdot \nu}{s^2} \\ \frac{\nu}{s} \end{bmatrix}$$

## 3. Síntese do Controlador (aula 15c)

O peso de desempenho $W_P(s)$ molda a função de sensibilidade:

$$W_P(s) = \frac{s/M + \omega_B}{s + \omega_B \cdot \varepsilon}$$

Parâmetros de Projeto:

*   $M = 2$: Limite de pico de sensibilidade (6 dB)
*   $\omega_B = 1$ rad/s: Largura de banda de malha fechada desejada
*   $\varepsilon = 0.01$: Limite de erro em regime permanente (1%)

Resposta em Frequência:

*   $|W_P(0)| = 100$ (40 dB)
*   $|W_P(\infty)| = 0.5$ (-6 dB)

#### $H_\infty$

Minimizar:

$$\gamma = \left\| \begin{bmatrix} W_P \cdot S \\ W_I \cdot K \cdot S \end{bmatrix} \right\|_\infty$$

Onde $S = (I + GK)^{-1}$ é a função de sensibilidade.


*   Primeira linha: Desempenho de rastreamento ($|W_P S|$ pequeno)
*   Segunda linha: Esforço de controle sob incerteza ($|W_I K S|$ pequeno)

#### Implementação via Loop-Shaping

Para o sistema de duplo integrador, alocamos os polos de malha fechada em:
$$s^2 + 2\zeta\omega_n s + \omega_n^2 = 0$$

Com $\zeta = 0.7$ (amortecimento ótimo) e $\omega_n = \omega_B / \zeta$.

**Ganhos Resultantes:**

```python
Kp_theta = 2 * zeta * omega_n / nu    # = 10.0
Kp_y = omega_n^2 / nu                  # = 10.2
Ki_theta = Kp_theta * omega_B / 10     # = 1.0
Ki_y = Kp_y * omega_B / 10             # = 1.02
```

Essa abordagem é compatível com o sistema de nós desenvolvido no Gazebo, permitindo a implementação direta do controlador MIMO com os ganhos calculados.
Porém, também seria possível utilizar a toolbox de controle robusto do Matlab, por exemplo. 

#### Espaço de Estados do Controlador

O controlador em tempo discreto:

$$x_c[k+1] = A_c \cdot x_c[k] + B_c \cdot e[k]$$
$$u[k] = C_c \cdot x_c[k] + D_c \cdot e[k]$$

Onde:

*   $x_c = [\int y, \int \theta]^T$ 
*   $e = [y_{error}, \theta_{error}]^T$ 
*   $u = [v, \omega]^T$ 

Matrizes:

```python
Ac = [[1, 0], [0, 1]]           # Integrador Discreto
Bc = [[0.2, 0], [0, 0.2]]       # dt =Ac = [[1, 0], [0, 1]]           # Integrador Discreto
Bc = [[0.2, 0], [0, 0.2]]       # dt = 0.2s
Cc = [[0, 0], [1.70, 1.0]]      # Ganhos Integrais
Dc = [[0, 0], [17.0, 10.0]]     # Ganhos Proporcionais


---



# Implementação

## 1. ROS

```mermaid
graph TD
    subgraph "Ground Truth"
        GT[rowfollow_gt.py] -->|heading_error| PP
        GT -->|distance_error| PP
    end
    
    subgraph "Path Processing"
        PP[reference_path_publisher.py] -->|/robot/path| MN
    end
    
    subgraph "Control"
        MN[control_node.py] -->|Controller Selection| RC
        MN -->|Controller Selection| Controller
        RC[RobustController] -->|cmd_vel| Robot
        Controller[ROBOT_CONTROLLER] -->|cmd_vel| Robot
    end
    
    subgraph "Monitoring"
        DL[control_data_logger.py] -->|CSV| Analysis
        DB[debug_mpc.py] -->|Reset Logic| Robot
    end
```

| Parameter | Value |
|-----------|-------|
| Environment | Gazebo |
| Reference velocity | 0.6 m/s |
| Sample time | 0.2 s |
| Duração | ~78 s |
| Ground truth | Direct state feedback |

## 2. Obtenção das Matrizes

1. Primeiro, um controlador PI simples em cada variável foi implementado. O código `control_data_logger.py` gravava os dados em um `.csv` para análise posterior.
2. O arquivo `analyze_control_data.py` exibiiu a resposta e parametros da simulação, utilizados diretametne para encontrar as matrizes do sistema linearizado.
3. Com os dados obtidos, as matrizes A, B, C, D foram calculadas conforme descrito na seção de modelagem da planta.

### Dados 

Resultado do analyze_control_data:

---

```text
# ANÁLISE DE DESEMPENHO

--- Resumo dos Dados ---
Duração:             77,80 s
Amostras:            779
Taxa de Amostragem:  10,0 Hz

--- Desempenho em Regime Permanente ---
Erro de Orientação (Heading): -0,166 ± 0,006 mrad
Erro de Distância:            -0,010 ± 0,063 mm
RMS de Rastreamento:          0,178 mm

--- Desempenho Transitório ---
Tempo de Acomodação: 0,60 s
Pico de Orientação:  4,60 mrad (0,26°)
Pico de Distância:   6,59 mm

--- Esforço de Controle ---
RMS do Cmd ω:        0,0055 rad/s
Máx. do Cmd ω:       0,0311 rad/s
Média do Cmd v:      0,600 m/s
RMS da Taxa do Cmd ω: 0,0035 rad/s²

# --- Para Síntese-μ (Mu-Synthesis) ---
Largura de Banda Est.: 0,03 Hz
Velocidade Média:      0,556 m/s

Parâmetros do MATLAB salvos em: /home/tommaselli/.ros/control_logs/plant_params_for_matlab.m

Gráfico salvo em: /home/tommaselli/.ros/control_logs/control_analysis.png
```

---

![Dados](dados.png)

Os ganhos finais foram calculados seguindo os passos descritos na seção de síntese do controlador.

# Resultados

## 1. PLANTA NOMINAL $G(s)$
**Parâmetros de Operação:**
* Velocidade ($v_{ref}$): `0.6 m/s`
* Skid angular ($\nu$): `0.2`

**Matrizes do Espaço de Estados:**
```math
A = [[0.0, 0.6], 
     [0.0, 0.0]]

B = [[0.0], 
     [0.2]]

C = [[1.0, 0.0], 
     [0.0, 1.0]]

D = [[0.0], 
     [0.0]]

```

**Funções de Transferência:**

* $G_y(s) = 0.12 / s^2$
* $G_\theta(s) = 0.2 / s$

---

## 2. CONTROLADOR 

Ganhos (Síntese $H_\infty$):
* $Kp_y = 10.2000$
* $Kp_\theta = 10.0000$
* $Ki_y = 1.0200$
* $Ki_\theta = 1.0000$

Função de Transferência ($\theta$):
* $K_\theta(s) = (10.0s + 1.0) / s$

---

## 3. PESOS DE SÍNTESE

**Peso de Desempenho** $W_P(s)$: 
- Função: $W_P = (s/2.0 + 1.0) / (s + 0.01)$
- Ganho DC: $|W_P(0)| = 100.0$ (40.0 dB)
- Alta Freq: $|W_P(\infty)| = 0.50$ (-6.0 dB)

**Peso de Incerteza $W_I(s)$** 
- Função: $W_I = (0.5s/10.0 + 0.1) / (s/10.0 + 1)$
- Incerteza DC: $|W_I(0)| = 0.10$ (10%)
- Incerteza Alta Freq: $|W_I(\infty)| = 0.50$ (50%)

---
### 4. ANÁLISE DE ESTABILIDADE E DESEMPENHO


**Estabilidade Nominal (NS)**

Polos de malha fechada:
- $p_1 = -1.8944$
- $p_2 = -0.1056$

Resposta:
- Frequência natural ($\omega_n$): 0.447 rad/s
- Amortecimento ($\zeta$): 2.236

> RESULTADO NS: ESTÁVEL

**Desempenho Nominal (NP)**

- Critério: $||W_P \cdot S||_\infty < 1$
- Valor: $0.5193$ (-5.69 dB)
- Pico em: $\omega = 0.871$ rad/s

> RESULTADO NP: SATISFEITO

**Estabilidade Robusta (RS)**

-Valor: $0.1049$ (-19.58 dB)
- Pico em: $\omega = 0.409$ rad/s
- Margem de estabilidade: 9.53 (Sistema tolera 953% da incerteza modelada)

> RESULTADO RS: GARANTIDO

**Desempenho Robusto (RP)**

- Critério SISO (conservador): $||W_P \cdot S|| + ||W_I \cdot T|| < 1$
- Valor Máximo: 0.6235
- Pico em: $\omega = 0.809$ rad/s

> RESULTADO RP: GARANTIDO

### Plots

![Resultado](resultado.png)
![Resultado](resultado2.png)
![Resultado](resultado3.png)

## RESUMO GERAL

| Métrica                | Status | Valor de Pico |
|------------------------|--------|---------------|
| Estabilidade Nominal   | Ok     | -             |
| Desempenho Nominal     | Ok     | 0.519         |
| Estabilidade Robusta   | Ok     | 0.105         |
| Desempenho Robusto     | Ok     | 0.624         |

- **Margem de Fase:** 87.1°  
- **Margem de Ganho:** ‑66.1 dB  
- **Frequência de Cruzamento:** 2.00 rad/s

# Conclusão 

Com a utilização da incerteza diretamente no projeto do controle, mesmo com as mudanças de terreno, essa interação foi minimizada, mantendo o desempenho desejado. O controlador MIMO projetado demonstrou robustez eficaz contra as variações paramétricas, garantindo estabilidade e desempenho satisfatórios ao longo da simulação.

# Código 

Para simplificar a leitura, o código completo está disponível no repositório GitHub: https://github.com/Felipe-Tommaselli/advanced-control/tree/main/mimo-control-systems/assigment4-mimo

Abaixo, o código foi limpo dos nós em ROS e os acessos ao resto do sistema. 

In [ ]:
"""
Controlador Linear Robusto para o Robô Terra

Implementa um controlador linear no espaço de estados
Baseado no modelo cinemático linearizado:
    ẏ = v_ref * θ       (dinâmica de erro lateral)
    θ̇ = ν * ω          (dinâmica de heading com coeficiente de derrapagem)

O controlador utiliza realimentação de estados com ação integral para erro nulo em regime permanente.
Ganhos podem ser sintetizados offline via métodos H∞/μ-síntese ou ajustados manualmente (PI).

Forma do controlador no espaço de estados:
    u[k] = Cc * x_c[k] + Dc * e[k]
    x_c[k+1] = Ac * x_c[k] + Bc * e[k]

Onde e = [erro_y, erro_θ]^T são os erros de rastreamento.
"""

import numpy as np
import os
import time

class RobustController:
    def __init__(self, params):
        """
        Inicializa o controlador linear robusto.
        
        params: dicionário com chaves:
            - v_ref: velocidade linear de referência
            - nu: coeficiente de derrapagem angular
            - dt: passo de tempo
            - v_lin_max: velocidade linear máxima
            - v_ang_max: velocidade angular máxima
            - Kp_y, Kp_theta: ganhos proporcionais
            - Ki_y, Ki_theta: ganhos integrais
        """
        # Parâmetros do Robô e Controle
        self.p = {
            'v_ref': params.get('v_ref', 0.6),
            'nu': params.get('nu', 0.2),
            'dt': params.get('dt', 0.2),
            'v_lin_max': params.get('v_lin_max', 1.0),
            'v_ang_max': params.get('v_ang_max', 1.0),
            'v_max': params.get('v_max', 1.0),     # Limite combinado das rodas
            'Lbase': params.get('Lbase', 0.27),
            'verbose': params.get('verbose', False),
            
            # Ganhos Padrão do Controlador PI
            'Kp_y': params.get('Kp_y', 2.0),
            'Kp_theta': params.get('Kp_theta', 1.5),
            'Ki_y': params.get('Ki_y', 0.1),
            'Ki_theta': params.get('Ki_theta', 0.05),
            
            # Opções de Síntese (ativar para usar matrizes H∞/μ)
            'use_synthesized_matrices': params.get('use_synthesized_matrices', False),
            'controller_matrices_file': params.get('controller_matrices_file', None)
        }
        
        # Configuração das matrizes do controlador
        if self.p['use_synthesized_matrices'] and self._load_synthesized_matrices():
            if self.p['verbose']: print("[RobustController] Matrizes sintetizadas carregadas.")
        else:
            if self.p['verbose']: print("[RobustController] Usando estrutura PI padrão.")
            self._setup_default_matrices()
        
        # Estados integrais (tamanho depende da ordem do controlador)
        self.x_c = np.zeros(self.Ac.shape[0])
        self.step_count = 0

    def _load_synthesized_matrices(self):
        """Carrega matrizes de controlador H∞/μ sintetizadas externamente (.npz)."""
        filepath = self.p['controller_matrices_file']
        if not filepath:
            filepath = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'robust_controller_matrices.npz')
        
        if not os.path.exists(filepath):
            return False
        
        try:
            data = np.load(filepath)
            self.Ac = data['Ac']
            self.Bc = data['Bc']
            self.Cc = data['Cc']
            self.Dc = data['Dc']
            return True
        except Exception as e:
            print(f"[RobustController] Erro ao carregar matrizes: {e}")
            return False

    def _setup_default_matrices(self):
        """
        Configura matrizes padrão para estrutura PI.
        x_c = [∫erro_y, ∫erro_θ]^T
        """
        dt = self.p['dt']
        v_ref = max(self.p['v_ref'], 0.1) # Evita divisão por zero
        
        # Integrador discreto: x_c[k+1] = x_c[k] + dt * e[k]
        self.Ac = np.eye(2)
        self.Bc = dt * np.eye(2)
        
        # Mapeamento Saída (u = Cc*x_c + Dc*e)
        # u = [v, ω]
        
        # Cc: Ganhos Integrais
        self.Cc = np.array([
            [0.0, 0.0],                                  # v: sem ação integral
            [self.p['Ki_y'] / v_ref, self.p['Ki_theta']]    # ω: termos integrais
        ])
        
        # Dc: Ganhos Proporcionais
        self.Dc = np.array([
            [0.0, 0.0],                                  # v: constante (adicionado depois)
            [self.p['Kp_y'] / v_ref, self.p['Kp_theta']]    # ω: termos proporcionais
        ])

    def reset(self):
        """Reseta os estados integrais do controlador."""
        self.x_c = np.zeros(self.Ac.shape[0])
        self.step_count = 0

    def updateParams(self, params):
        """Atualiza parâmetros dinamicamente e recalcula matrizes se necessário."""
        updated = False
        for k, v in params.items():
            if k in self.p and self.p[k] != v:
                self.p[k] = v
                updated = True
        
        if updated and not self.p['use_synthesized_matrices']:
            self._setup_default_matrices()

    def solve(self, reference, omega=0):
        """
        Calcula a ação de controle.
        
        Args:
            reference: dict com 'x', 'y', 'theta' (erros relativos no frame do robô)
            omega: velocidade angular atual (não usado neste controlador linear)
            
        Returns:
            u: controle [v, ω] (2xN)
            pred_vals: estados previstos (simplificado)
            ss_error: erro de estado estacionário
        """
        self.step_count += 1
        
        # Se não há referência, parar
        if len(reference['x']) == 0:
            return np.zeros((2, 15)), np.zeros((3, 16)), 0.0
        
        # Extração de Erros (y[0] e theta[0] são os erros atuais no frame do robô)
        y_error = reference['y'][0]
        theta_error = reference['theta'][0]
        e = np.array([y_error, theta_error])
        
        # Lei de Controle no Espaço de Estados: u = Cc*x_c + Dc*e
        u_raw = self.Cc @ self.x_c + self.Dc @ e
        
        # Adiciona feedforward da velocidade de referência
        v_cmd = self.p['v_ref'] + u_raw[0]
        omega_cmd = u_raw[1]
        
        # Saturações
        omega_cmd = np.clip(omega_cmd, -self.p['v_ang_max'], self.p['v_ang_max'])
        v_cmd = np.clip(v_cmd, -self.p['v_lin_max'], self.p['v_lin_max'])
        
        # v ± (L/2)*ω <= v_max
        v_left = v_cmd + 0.5 * self.p['Lbase'] * omega_cmd
        v_right = v_cmd - 0.5 * self.p['Lbase'] * omega_cmd
        max_wheel = max(abs(v_left), abs(v_right))
        
        if max_wheel > self.p['v_max']:
            scale = self.p['v_max'] / max_wheel
            v_cmd *= scale
            omega_cmd *= scale
        
        # Atualização dos Estados (Dinâmica do Controlador)
        # x_c[k+1] = Ac*x_c + Bc*e
        self.x_c = self.Ac @ self.x_c + self.Bc @ e
        
        self.x_c = np.clip(self.x_c, -10.0, 10.0)
        
        # Logging Opcional
        if self.p['verbose'] and self.step_count % 20 == 0:
            print(f"[RobustCtrl] Erro Y: {y_error:.3f}, Erro Th: {theta_error:.3f} | Cmd: v={v_cmd:.2f}, w={omega_cmd:.2f}")

        # Formatação de Saída 
        N = len(reference['x']) - 1 if len(reference['x']) > 1 else 15
        u = np.zeros((2, N))
        u[0, :] = v_cmd  # Mantém controle constante no horizonte
        u[1, :] = omega_cmd
        
        # Predição simplificada (apenas para visualização/compatibilidade)
        pred_vals = np.zeros((3, N + 1))
        ss_error = np.sqrt(reference['x'][0]**2 + reference['y'][0]**2)
        
        return u, pred_vals, ss_error